# Chat Agent

### State Diagram (Agent View)
```mermaid
stateDiagram-v2
direction LR
INIT --> CHAT
CHAT --> FINAL
```


### a) Create Agent

In [1]:
from gai.asm.agents import ChatAgent
from gai.mcp.client.mcp_client import McpAggregatedClient
from gai.lib.config import config_helper

from gai.lib.tests import make_local_tmp
import os
here = make_local_tmp()
file_path = os.path.join(here, "monologue.json")
from gai.messages import FileMonologue
monologue = FileMonologue(agent_name="ChatAgent",file_path=file_path)

aggregated_client = McpAggregatedClient(["mcp-pseudo","mcp-time", "mcp-web"])
tools = await aggregated_client.list_tools()
agent = ChatAgent(
    agent_name="ChatAgent",
    llm_config=config_helper.get_client_config(
        {
            "client_type": "anthropic",
            "model": "claude-sonnet-4-20250514",
            "extra": {
                "max_tokens": 32000,
                "temperature": 0.7,
                "top_p": 0.95,
                "tools": True,
                "stream": True,
            },
        }
    ),
    monologue=monologue
)

### reset monologue (optional)

In [2]:
monologue.reset()


### b) run_async

In [3]:
user_message="Tell me a one paragraph story."
resp=await agent.run_async(user_message=user_message)
# Stream the response
async for chunk in resp():
    if chunk:
        if isinstance(chunk, str):
            print(chunk, end="", flush=True)

The old lighthouse keeper discovered that every night at exactly 3:17 AM, a ship would appear on the horizon—the same ship that had vanished in a storm thirty years ago, taking his brother with it. For weeks, he watched through his telescope as the ghostly vessel sailed the same path, its crew moving about the deck in silent routine, until one night he realized they weren't lost souls at all, but patient guardians, steering modern ships away from the hidden rocks that had claimed their own lives. With tears streaming down his weathered face, he flashed the lighthouse beam three times in gratitude, and for the first time in three decades, his brother's ship flashed back before fading into the dawn, its duty finally acknowledged.

### c) Show monologue

In [4]:
import json

# Show the monologue
print("\n───────────────────────── MONOLOGUE START ─────────────────────────")
messages = agent.fsm.monologue.list_messages()
for message in messages[-2:]:
    print(json.dumps(message.model_dump(), indent=4))
print("───────────────────────── MONOLOGUE END ─────────────────────────\n")

# Print memory size
mem_size = agent.fsm.monologue.get_total_size()
print("Total char size=", mem_size)


───────────────────────── MONOLOGUE START ─────────────────────────
{
    "id": "d056202d-24c4-44d4-a789-d3f30c0b2a0c",
    "header": {
        "sender": "User",
        "recipient": "ChatAgent",
        "timestamp": 1752391146.8062885,
        "order": 0
    },
    "body": {
        "type": "monologue",
        "state_name": "CHAT",
        "step_no": 1,
        "content_type": "text",
        "role": "user",
        "content": "Tell me a one paragraph story."
    }
}
{
    "id": "941e134e-3316-4c34-925d-ff361d86f9e8",
    "header": {
        "sender": "ChatAgent",
        "recipient": "User",
        "timestamp": 1752391153.8444932,
        "order": 1
    },
    "body": {
        "type": "monologue",
        "state_name": "CHAT",
        "step_no": 1,
        "content_type": "text",
        "role": "assistant",
        "content": [
            {
                "citations": null,
                "text": "The old lighthouse keeper discovered that every night at exactly 3:17 AM, a shi

Confirm if its end of response by continuing the agent.

In [5]:
resp = await agent.run_async()
# Stream the response
async for chunk in resp():
    if chunk:
        if isinstance(chunk, str):
            print(chunk, end="", flush=True)


TypeError: ChatAgent.run_async() missing 1 required positional argument: 'user_message'